# Flight Data Ingestion: S3, Athena, and SageMaker
**AAI-540**

**Team 1**

**Thomas Geraci**

## 1. Imports

In [23]:
import csv
import gzip
import hashlib
import io
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import pandas as pd
from botocore.exceptions import ClientError, NoCredentialsError
from IPython.display import display

## 2. Configuration 

In [24]:
print("Notebook revision 4: SSE-S3 encryption and .csv.gz support")
AWS_REGION = "us-east-1"  #Change to your SageMaker region if different.
AWS_PROFILE = None        
DATA_YEAR = 2025
LOCAL_RAW_DIR = Path("data/raw")
RAW_BUCKET = "aai-540-group1-flight-data-raw"
PROCESSED_BUCKET = "aai-540-group1-flight-data-processed"
CREATE_MISSING_BUCKETS = True
ATHENA_WORKGROUP = "primary"  
ATHENA_DATABASE = "aai540_group1"
RAW_TABLE = "bts_raw_2025"
TYPED_VIEW = "bts_ingestion_2025"
RAW_PREFIX = f"bts/{DATA_YEAR}/"
RESULT_PREFIX = "athena-results/"
REPORT_PREFIX = f"ingestion-reports/{DATA_YEAR}/"
QUERY_TIMEOUT_SECONDS = 600
LOCAL_REPORT_DIR = Path("reports/ingestion")

for identifier in (ATHENA_DATABASE, RAW_TABLE, TYPED_VIEW):
    if not re.fullmatch(r"[a-z][a-z0-9_]*", identifier):
        raise ValueError(f"Use a lowercase SQL identifier: {identifier}")
for bucket in (RAW_BUCKET, PROCESSED_BUCKET):
    if not re.fullmatch(r"[a-z0-9][a-z0-9.-]{1,61}[a-z0-9]", bucket):
        raise ValueError(f"Invalid S3 bucket name: {bucket}")
if RAW_BUCKET == PROCESSED_BUCKET:
    raise ValueError("Keep raw and processed data in separate buckets, as designed.")

Notebook revision 4: SSE-S3 encryption and .csv.gz support


## 3. Confirm AWS identity and connection

In [25]:
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
try:
    identity = session.client("sts").get_caller_identity()
except NoCredentialsError as exc:
    raise RuntimeError("No AWS credentials. Open this in SageMaker with an attached execution role.") from exc
ACCOUNT_ID = identity["Account"]
s3 = session.client("s3")
athena = session.client("athena")
glue = session.client("glue")
print(f"Connected to AWS in {AWS_REGION}; credentials available.")
print("Capture AWS identity privately if needed; do not publish credentials.")

Connected to AWS in us-east-1; credentials available.
Capture AWS identity privately if needed; do not publish credentials.


## 4. Create S3 data lake

In [26]:
def ensure_bucket(bucket):
    created = False
    try:
        s3.head_bucket(Bucket=bucket, ExpectedBucketOwner=ACCOUNT_ID)
    except ClientError as exc:
        error = str(exc.response["Error"]["Code"])
        if error not in {"404", "NoSuchBucket", "NotFound"}:
            raise RuntimeError(f"Cannot access {bucket}; check ownership and role permissions.") from exc
        if not CREATE_MISSING_BUCKETS:
            raise RuntimeError(f"Create {bucket} in {AWS_REGION}, or enable CREATE_MISSING_BUCKETS.") from exc
        args = {"Bucket": bucket}
        if AWS_REGION != "us-east-1":
            args["CreateBucketConfiguration"] = {"LocationConstraint": AWS_REGION}
        s3.create_bucket(**args)
        s3.put_public_access_block(
            Bucket=bucket, ExpectedBucketOwner=ACCOUNT_ID,
            PublicAccessBlockConfiguration={
                "BlockPublicAcls": True, "IgnorePublicAcls": True,
                "BlockPublicPolicy": True, "RestrictPublicBuckets": True,
            },
        )
        s3.put_bucket_encryption(
            Bucket=bucket, ExpectedBucketOwner=ACCOUNT_ID,
            ServerSideEncryptionConfiguration={"Rules": [{
                "ApplyServerSideEncryptionByDefault": {
                    "SSEAlgorithm": "AES256",
                },
            }]},
        )
        s3.put_bucket_versioning(Bucket=bucket, ExpectedBucketOwner=ACCOUNT_ID,
                                 VersioningConfiguration={"Status": "Enabled"})
        created = True
    actual_region = s3.get_bucket_location(Bucket=bucket, ExpectedBucketOwner=ACCOUNT_ID)["LocationConstraint"]
    actual_region = "us-east-1" if actual_region is None else ("eu-west-1" if actual_region == "EU" else actual_region)
    if actual_region != AWS_REGION:
        raise RuntimeError(f"{bucket} is in {actual_region}; change configuration to use a consistent region.")
    public = s3.get_public_access_block(Bucket=bucket, ExpectedBucketOwner=ACCOUNT_ID)["PublicAccessBlockConfiguration"]
    if not all(public.get(k, False) for k in ("BlockPublicAcls", "IgnorePublicAcls", "BlockPublicPolicy", "RestrictPublicBuckets")):
        raise RuntimeError(f"Enable all four Block Public Access settings on {bucket} before continuing.")
    encryption = s3.get_bucket_encryption(Bucket=bucket, ExpectedBucketOwner=ACCOUNT_ID)
    algorithms = [r["ApplyServerSideEncryptionByDefault"]["SSEAlgorithm"]
                  for r in encryption["ServerSideEncryptionConfiguration"]["Rules"]]
    if algorithms != ["AES256"]:
        raise RuntimeError(f"Expected SSE-S3 on {bucket} for the separate-account lab implementation. Review bucket encryption.")
    return {"bucket": bucket, "region": actual_region, "created": created,
            "public_access_blocked": True, "encryption": "SSE-S3"}

bucket_checks = [ensure_bucket(b) for b in (RAW_BUCKET, PROCESSED_BUCKET)]
display(pd.DataFrame(bucket_checks))

,bucket,region,created,public_access_blocked,encryption
0,aai-540-group1-flight-data-raw,us-east-1,False,True,SSE-S3
1,aai-540-group1-flight-data-processed,us-east-1,False,True,SSE-S3


## 5. Inspect local CSVs and upload original content

In [27]:
def read_csv_header(path):
    opener = gzip.open if path.name.endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8-sig", newline="") as stream:
        return next(csv.reader(stream))

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

local_csvs = sorted(p for p in LOCAL_RAW_DIR.rglob("*") if p.is_file()
                   and (p.name.lower().endswith(".csv") or p.name.endswith(".csv.gz")))
logical_names = [p.name.removesuffix(".gz").lower() for p in local_csvs]
if len(set(logical_names)) != len(logical_names):
    raise ValueError("Choose either plain CSV or gzip for each month, with no duplicate source filenames.")
local_headers = [read_csv_header(p) for p in local_csvs]
if local_headers and any(h != local_headers[0] for h in local_headers):
    raise ValueError("CSV headers differ. Redownload each month with the same fields and column order.")
if len({p.name for p in local_csvs}) != len(local_csvs):
    raise ValueError("Duplicate CSV filenames in subfolders would collide in S3. Use unique source names.")

upload_manifest = []
for path in local_csvs:
    key = RAW_PREFIX + path.name
    digest = file_sha256(path)
    try:
        existing = s3.head_object(Bucket=RAW_BUCKET, Key=key, ExpectedBucketOwner=ACCOUNT_ID)
    except ClientError as exc:
        if str(exc.response["Error"]["Code"]) not in {"404", "NoSuchKey", "NotFound"}:
            raise
        existing = None
    if existing is not None:
        if existing.get("Metadata", {}).get("sha256") != digest or existing["ContentLength"] != path.stat().st_size:
            raise RuntimeError(f"Existing raw object differs or has no provenance hash: {key}. Review it before proceeding.")
        status = "already_uploaded"
    else:
        s3.upload_file(str(path), RAW_BUCKET, key, ExtraArgs={
            "ExpectedBucketOwner": ACCOUNT_ID, "ServerSideEncryption": "AES256",
            "ContentType": "application/gzip" if path.name.endswith(".gz") else "text/csv",
            "Metadata": {"sha256": digest, "source": "bts-transtats", "year": str(DATA_YEAR)},
        })
        status = "uploaded"
    upload_manifest.append({"filename": path.name, "s3_uri": f"s3://{RAW_BUCKET}/{key}",
                            "bytes": path.stat().st_size, "sha256": digest, "status": status})
if upload_manifest:
    display(pd.DataFrame(upload_manifest))
else:
    print(f"No local CSVs in {LOCAL_RAW_DIR.resolve()}. Checking for data already in S3 next.")

No local CSVs in /home/sagemaker-user/aai-540-homework/data/raw. Checking for data already in S3 next.


## 6. Verify S3 inventory and derive the actual schema

In [28]:
objects = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=RAW_BUCKET, Prefix=RAW_PREFIX,
                                                        ExpectedBucketOwner=ACCOUNT_ID):
    objects.extend(o for o in page.get("Contents", []) if not o["Key"].endswith("/"))
if not objects:
    raise RuntimeError(f"No data at s3://{RAW_BUCKET}/{RAW_PREFIX}. Upload extracted BTS CSVs to data/raw/ and rerun Section 5.")
if any(not (o["Key"].lower().endswith(".csv") or o["Key"].endswith(".csv.gz")) for o in objects):
    raise RuntimeError("Use only .csv or .csv.gz data with lowercase .gz suffix. Move ZIPs/reports/lookup tables elsewhere.")
logical_keys = [o["Key"].removesuffix(".gz").lower() for o in objects]
if len(set(logical_keys)) != len(logical_keys):
    raise ValueError("A CSV and its gzip copy are both present. Retain one representation per monthly file to avoid counting flights twice.")

def s3_csv_header(key):
    response = s3.get_object(Bucket=RAW_BUCKET, Key=key, ExpectedBucketOwner=ACCOUNT_ID)
    with response["Body"] as body:
        binary = gzip.GzipFile(fileobj=body, mode="rb") if key.endswith(".gz") else body
        with io.TextIOWrapper(binary, encoding="utf-8-sig", newline="") as stream:
            return next(csv.reader(stream))

header = s3_csv_header(objects[0]["Key"])
for obj in objects[1:]:
    if s3_csv_header(obj["Key"]) != header:
        raise ValueError(f"S3 schema mismatch: {obj['Key']}. Every file must use identical columns in identical order.")

def normalize_header(name, index):
    normalized = re.sub(r"[^a-z0-9]", "", name.lower())
    if not normalized:
        normalized = f"unnamed{index}"
    if normalized[0].isdigit():
        normalized = "col" + normalized
    return normalized

columns = [normalize_header(name, i) for i, name in enumerate(header)]
if len(set(columns)) != len(columns):
    raise ValueError("Headers collide after normalization. Inspect the source schema before continuing.")
schema_df = pd.DataFrame({"position": range(len(header)), "bts_header": header,
                          "athena_column": columns, "raw_type": "string"})
inventory_df = pd.DataFrame([{"key": o["Key"], "bytes": o["Size"]} for o in objects])
display(inventory_df)
display(schema_df)
print(f"Raw files: {len(objects):,}; columns including any blank trailing column: {len(columns):,}")

,key,bytes
0,bts/2025/April.csv.gz,30989463
1,bts/2025/August.csv.gz,32345537
2,bts/2025/December.csv.gz,31575829
3,bts/2025/Febuary.csv.gz,26853819
4,bts/2025/January.csv.gz,28456705
5,bts/2025/July.csv.gz,34346608
6,bts/2025/June.csv.gz,33228721
7,bts/2025/March.csv.gz,31919159
8,bts/2025/May.csv.gz,32496556
9,bts/2025/November.csv.gz,30381105


,position,bts_header,athena_column,raw_type
0,0,YEAR,year,string
1,1,QUARTER,quarter,string
2,2,MONTH,month,string
3,3,DAY_OF_MONTH,dayofmonth,string
4,4,DAY_OF_WEEK,dayofweek,string
...,...,...,...,...
103,103,DIV5_WHEELS_ON,div5wheelson,string
104,104,DIV5_TOTAL_GTIME,div5totalgtime,string
105,105,DIV5_LONGEST_GTIME,div5longestgtime,string
106,106,DIV5_WHEELS_OFF,div5wheelsoff,string


Raw files: 12; columns including any blank trailing column: 108


## 7. Athena execution helper

In [29]:
workgroup = athena.get_work_group(WorkGroup=ATHENA_WORKGROUP)["WorkGroup"]
wg_config = workgroup["Configuration"]
print("Workgroup:", ATHENA_WORKGROUP)
print("Engine:", wg_config.get("EngineVersion", {}))
print("Workgroup enforces configuration:", wg_config.get("EnforceWorkGroupConfiguration", False))
print("Workgroup result settings:", wg_config.get("ResultConfiguration", {}))

query_log = []
def run_query(sql, database=None):
    args = {
        "QueryString": sql, "WorkGroup": ATHENA_WORKGROUP,
        "ResultConfiguration": {
            "OutputLocation": f"s3://{PROCESSED_BUCKET}/{RESULT_PREFIX}",
            "ExpectedBucketOwner": ACCOUNT_ID,
            "EncryptionConfiguration": {"EncryptionOption": "SSE_S3"},
        },
    }
    if database is not None:
        args["QueryExecutionContext"] = {"Database": database, "Catalog": "AwsDataCatalog"}
    query_id = athena.start_query_execution(**args)["QueryExecutionId"]
    deadline = time.monotonic() + QUERY_TIMEOUT_SECONDS
    while True:
        execution = athena.get_query_execution(QueryExecutionId=query_id)["QueryExecution"]
        status = execution["Status"]["State"]
        if status in {"SUCCEEDED", "FAILED", "CANCELLED"}:
            break
        if time.monotonic() >= deadline:
            athena.stop_query_execution(QueryExecutionId=query_id)
            raise TimeoutError(f"Athena query {query_id} timed out and was cancelled.")
        time.sleep(2)
    query_log.append({"query_id": query_id, "state": status,
                      "bytes_scanned": execution.get("Statistics", {}).get("DataScannedInBytes", 0),
                      "sql": sql, "result_configuration": execution.get("ResultConfiguration", {})})
    if status != "SUCCEEDED":
        raise RuntimeError(f"Athena {status}: {execution['Status'].get('StateChangeReason', '')}")
    rows, names = [], None
    for page in athena.get_paginator("get_query_results").paginate(QueryExecutionId=query_id):
        result = page["ResultSet"]
        page_rows = result.get("Rows", [])
        if names is None:
            names = [c["Name"] for c in result["ResultSetMetadata"]["ColumnInfo"]]
            page_rows = page_rows[1:]  # Athena includes one header row on the first page.
        for row in page_rows:
            rows.append([c.get("VarCharValue") for c in row["Data"]])
    return pd.DataFrame(rows, columns=names or [])

Workgroup: primary
Engine: {'SelectedEngineVersion': 'AUTO', 'EffectiveEngineVersion': 'Athena engine version 3'}
Workgroup enforces configuration: False
Workgroup result settings: {}


## 8. Create the database and raw table

In [30]:
run_query(f"CREATE DATABASE IF NOT EXISTS {ATHENA_DATABASE}")
column_ddl = ",\n".join(f"  `{name}` string" for name in columns)
create_table_sql = f'''CREATE EXTERNAL TABLE IF NOT EXISTS `{ATHENA_DATABASE}`.`{RAW_TABLE}` (
{column_ddl}
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES ('separatorChar'=',', 'quoteChar'='"')
STORED AS TEXTFILE
LOCATION 's3://{RAW_BUCKET}/{RAW_PREFIX}'
TBLPROPERTIES ('skip.header.line.count'='1')'''
run_query(create_table_sql, ATHENA_DATABASE)
metadata = glue.get_table(DatabaseName=ATHENA_DATABASE, Name=RAW_TABLE)["Table"]["StorageDescriptor"]
actual_schema = [(c["Name"], c["Type"]) for c in metadata["Columns"]]
if actual_schema != [(c, "string") for c in columns] or metadata["Location"].rstrip("/") != f"s3://{RAW_BUCKET}/{RAW_PREFIX}".rstrip("/"):
    raise RuntimeError("Existing Athena table differs. Choose a new RAW_TABLE name or coordinate a metadata correction with the team.")
print(f"Catalog validated: {ATHENA_DATABASE}.{RAW_TABLE}")
display(run_query(f'SELECT * FROM "{RAW_TABLE}" LIMIT 5', ATHENA_DATABASE))


Catalog validated: aai540_group1.bts_raw_2025


,year,quarter,month,dayofmonth,dayofweek,fldate,opuniquecarrier,opcarrierairlineid,opcarrier,tailnum,...,div4wheelsoff,div4tailnum,div5airport,div5airportid,div5airportseqid,div5wheelson,div5totalgtime,div5longestgtime,div5wheelsoff,div5tailnum
0,2025,4,12,1,1,12/1/2025 12:00:00 AM,AA,19805,AA,N101NN,...,,,,,,,,,,
1,2025,4,12,1,1,12/1/2025 12:00:00 AM,AA,19805,AA,N101NN,...,,,,,,,,,,
2,2025,4,12,1,1,12/1/2025 12:00:00 AM,AA,19805,AA,N101NN,...,,,,,,,,,,
3,2025,4,12,1,1,12/1/2025 12:00:00 AM,AA,19805,AA,N102NN,...,,,,,,,,,,
4,2025,4,12,1,1,12/1/2025 12:00:00 AM,AA,19805,AA,N102UW,...,,,,,,,,,,


## 9. Create a typed inspection view

In [31]:
aliases = {
    "flight_date": ["fldate", "flightdate"],
    "carrier": ["opuniquecarrier", "reportingairline", "uniquecarrier"],
    "origin": ["origin"], "destination": ["dest"],
    "scheduled_departure": ["crsdeptime"], "scheduled_arrival": ["crsarrtime"],
    "distance": ["distance"], "cancelled": ["cancelled"],
    "diverted": ["diverted"], "arrival_delay": ["arrdelay"],
}
resolved = {}
for canonical, options in aliases.items():
    matches = [name for name in options if name in columns]
    if not matches:
        raise ValueError(f"Missing {canonical}; expected one of {options}. Redownload all months with required fields.")
    resolved[canonical] = matches[0]

def value(name):
    return f"NULLIF(TRIM(\"{resolved[name]}\"), '')"

date_value = value("flight_date")
date_token = f"SPLIT_PART({date_value}, ' ', 1)"
expressions = [
    f"COALESCE(TRY_CAST(SUBSTR({date_value}, 1, 10) AS DATE), "
    f"CAST(TRY(date_parse({date_token}, '%c/%e/%Y')) AS DATE)) AS flight_date",
    *[f"{value(name)} AS {name}" for name in ("carrier", "origin", "destination")],
    *[f"TRY_CAST({value(name)} AS DOUBLE) AS {name}" for name in
      ("scheduled_departure", "scheduled_arrival", "distance", "cancelled", "diverted", "arrival_delay")],
]
view_sql = (f'CREATE OR REPLACE VIEW "{TYPED_VIEW}" AS SELECT\n'
            + ",\n".join(expressions) + f'\nFROM "{RAW_TABLE}"')
run_query(view_sql, ATHENA_DATABASE)
display(run_query(f'SELECT * FROM "{TYPED_VIEW}" LIMIT 5', ATHENA_DATABASE))

,flight_date,carrier,origin,destination,scheduled_departure,scheduled_arrival,distance,cancelled,diverted,arrival_delay
0,2025-12-01,AA,JFK,LAX,700.0,1035.0,2475.0,0.0,0.0,-5.0
1,2025-12-01,AA,JFK,LAX,2100.0,28.0,2475.0,0.0,0.0,-36.0
2,2025-12-01,AA,LAX,JFK,1135.0,2000.0,2475.0,0.0,0.0,-8.0
3,2025-12-01,AA,LAX,JFK,605.0,1430.0,2475.0,0.0,0.0,352.0
4,2025-12-01,AA,BUF,CLT,1913.0,2115.0,546.0,0.0,0.0,13.0


## 10. Count records, check year coverage, and inspect label eligibility

In [32]:
eligible = "cancelled = 0 AND diverted = 0 AND arrival_delay IS NOT NULL"
profile_sql = f'''SELECT
    COUNT(*) AS total_records,
    CAST(MIN(flight_date) AS VARCHAR) AS first_flight_date,
    CAST(MAX(flight_date) AS VARCHAR) AS last_flight_date,
    COUNT_IF(flight_date IS NULL) AS invalid_or_missing_dates,
    COUNT_IF(flight_date IS NOT NULL AND YEAR(flight_date) <> {DATA_YEAR}) AS wrong_year_records,
    COUNT_IF(cancelled = 1) AS cancelled_records,
    COUNT_IF(diverted = 1) AS diverted_records,
    COUNT_IF(arrival_delay IS NULL) AS missing_or_invalid_arrival_delay,
    COUNT_IF(cancelled IS NULL OR diverted IS NULL OR cancelled NOT IN (0,1) OR diverted NOT IN (0,1)) AS invalid_status_records,
    COUNT_IF({eligible} AND YEAR(flight_date) = {DATA_YEAR}) AS preliminary_label_eligible_records
FROM "{TYPED_VIEW}"'''
profile_df = run_query(profile_sql, ATHENA_DATABASE)
display(profile_df)
total_records = int(profile_df.iloc[0]["total_records"])
if total_records == 0:
    raise ValueError("The table has no flight rows. Check CSV contents and the raw prefix.")
if int(profile_df.iloc[0]["wrong_year_records"]) > 0:
    raise ValueError("Other years exist in the raw prefix. Correct the inventory before handing off.")

monthly_df = run_query(f'''SELECT MONTH(flight_date) AS month, COUNT(*) AS records
FROM "{TYPED_VIEW}" WHERE YEAR(flight_date) = {DATA_YEAR}
GROUP BY MONTH(flight_date) ORDER BY month''', ATHENA_DATABASE)
display(monthly_df)
months_present = sorted(int(m) for m in monthly_df["month"])
full_year_coverage = months_present == list(range(1, 13))
print("All 12 months represented:", full_year_coverage)
if not full_year_coverage:
    print("Starter ingestion only: add missing months, then rerun. Do not report full-year ingestion complete.")
else:
    print("Month coverage is complete; verify each monthly source download was complete before reporting full-year ingestion.")

label_df = run_query(f'''SELECT
CASE WHEN arrival_delay >= 15 THEN 1 ELSE 0 END AS delayed,
COUNT(*) AS records
FROM "{TYPED_VIEW}"
WHERE {eligible} AND YEAR(flight_date) = {DATA_YEAR}
GROUP BY CASE WHEN arrival_delay >= 15 THEN 1 ELSE 0 END ORDER BY delayed''', ATHENA_DATABASE)
label_df["records"] = pd.to_numeric(label_df["records"])
label_df["share"] = label_df["records"] / label_df["records"].sum()
display(label_df)
class_counts = {int(row["delayed"]): int(row["records"]) for _, row in label_df.iterrows()}
minimum_class_volume_met = all(class_counts.get(label, 0) >= 10_000 for label in (0, 1))
print("At least 10,000 preliminary eligible records per class:", minimum_class_volume_met)
print("These are preliminary counts before cleaning; missing/cancelled/diverted categories can overlap.")

,total_records,first_flight_date,last_flight_date,invalid_or_missing_dates,wrong_year_records,cancelled_records,diverted_records,missing_or_invalid_arrival_delay,invalid_status_records,preliminary_label_eligible_records
0,7001619,2025-01-01,2025-12-31,0,0,102876,19258,122135,0,6879484


,month,records
0,1,539747
1,2,504884
2,3,600872
3,4,583950
4,5,605648
5,6,611575
6,7,631428
7,8,602378
8,9,562439
9,10,605844


All 12 months represented: True
Month coverage is complete; verify each monthly source download was complete before reporting full-year ingestion.


,delayed,records,share
0,0,5344846,0.776925
1,1,1534638,0.223075


At least 10,000 preliminary eligible records per class: True
These are preliminary counts before cleaning; missing/cancelled/diverted categories can overlap.
